# 期待収益性の検証:将来の収益性の予測、Fama-MacBeth 回帰、RMW6(実績 vs 期待)

参考論文:Wahal (2025) "Expected Profitability"

4つの収益性指標について、論文と同じ流れで「過去の情報から将来の収益性をどれだけ予測できるか」「実績と期待のどちらがリターンをよく説明するか」
「実績と期待のどちらでファクターを作るとパフォーマンスがよいか」を検証します。結果は **2つのグループに分けて** 示します。

| グループ | 指標 | ねらい |
|---|---|---|
| **キャッシュベース** | CBOP・CP | 論文で使われている、アクルーアルを調整した収益性 |
| **なじみのある指標** | ROA・ROE | 実務で一般的な収益性。過去の情報からどれだけ予測でき、リターンをどれだけ説明できるか |

| 章 | 内容 |
|---|---|
| 0〜3 | 設定、取得関数、データ整形、期待収益性の推定(`rmw18_factor_analysis.ipynb` と同じコード) |
| 4 | 月次データ(リターン、モメンタム、バイアンドホールドのウェイト) |
| 5 | 将来の収益性の予測(論文 Table 1 に相当)と予測精度 |
| 6 | Fama-MacBeth 回帰(実績 vs 期待) |
| 7 | RMW6(規模 × 収益性の 2×3)のファクターリターン:実績 vs 期待 |
| 8 | まとめ |
| 9 | 結果の保存 |

## メソドロジー

### 収益性指標とデータの時点

| 指標 | 分子(当期 t) | 分母(前期末 t−1) | 投資の変数 |
|---|---|---|---|
| CBOP | 営業利益 + 研究開発費 − アクルーアル | 総資産 | 総資産成長率 |
| CP | 営業利益 − 支払利息 − アクルーアル | 自己資本 | 自己資本成長率 |
| ROA | 営業利益 | 総資産 | 総資産成長率 |
| ROE | 当期純利益 | 自己資本 | 自己資本成長率 |

- 本決算のみ。分子と分母は1期ずらし、同じ会計基準でそろえる(複数ある場合は SEC → IFRS → 日本)
- 毎年6月末に、6月末までに公表された本決算で計算する。B/M = 自己資本 ÷ 決算期末の時価総額、規模 = 6月末の時価総額
- 投資は1期前の成長率(t−2 → t−1)。自己資本成長率は同じ項目どうしで計算し、2006年の会社法施行で項目が切り替わった年などは自己資本どうしで計算
- ユニバース:TOPIX 構成銘柄(電気・ガス・金融を除く)。マイクロキャップ = ユニバースの時価総額 20 パーセンタイル未満

### 期待収益性(重回帰のみ)

$$Y_{t+1} = \alpha + \beta_1 Y_t + \beta_2 \Delta Y_t + \gamma_1 \ln(B/M)_t + \gamma_2 \ln(ME)_t + \gamma_3 Inv_t + \varepsilon$$

- 毎年、前年のコホートのうち、今年6月末までに翌期の実績が公表された企業(マイクロキャップを除く)で推定し、
  その係数を今年の説明変数に当てはめて Yhat とする(先読みなし)
- 過去の収益性だけを使った単回帰のモデルは使わない

### 5章:将来の収益性の予測

翌期の収益性 $Y_{t+1}$ を、3つの重回帰(いずれも ln(B/M)・ln(ME)・投資をコントロール)で予測し、年次回帰の係数の平均と Newey-West t 値、
自由度調整済み R² の平均を比べます(マイクロキャップを除く共通サンプル)。

| 式 | 収益性の説明変数 |
|---|---|
| (A) | $Y_t$ |
| (B) | $Y_t$, $Y_{t-1}$ |
| (C) | $Y_t$, $\Delta Y_t$(= 期待収益性の式) |

偏 R² = (R²(C) − R²(A)) / (1 − R²(A)) で、過去の収益性の変化を加えたことによる説明力の上乗せを測ります。
あわせて、実績 $Y_t$ をそのまま予測値とする場合(ランダムウォーク)と、期待 Yhat(サンプル外)の予測精度(順位相関・MAE・R²)を比べます。

### 6章:Fama-MacBeth 回帰

毎月、月次リターンを収益性とコントロール(ln(B/M)、ln(ME)、投資、前月リターン R(1,1)、t−12〜t−2 の11か月リターン R(2,12))に回帰し、
係数の時系列平均(×100)と Newey-West t 値(ラグ12)を出します。全ての式で同じサンプルを使います。

| 式 | 収益性の変数 |
|---|---|
| (1) | 実績 $Y_t$ |
| (2) | $Y_t$, $\Delta Y_t$ |
| (3) | $Y_t$, 期待 Yhat |
| (4) | 期待 Yhat |

- ユニバース:全銘柄 / マイクロキャップ除く。加重:等加重(OLS)/ 時価総額加重(WLS、ウェイトはバイアンドホールド)
- IQR リターン差 = 毎月の係数 × その月の変数の四分位範囲 の平均(%/月)。(4) と (1) の差の t 値も出す

### 7章:RMW6(実績 vs 期待)

毎年6月末に、時価総額の累積 90% までを大型、残りを小型とし、大型・小型それぞれの中で収益性(実績 Y または期待 Yhat)の
30・70 パーセンタイルで低・中・高に分けた 2×3 = 6 ポートフォリオを作り、7月〜翌6月までバイアンドホールドで保有します(各ポートフォリオ内は時価総額加重)。

$$\text{RMW6} = \tfrac{1}{2}\,(\text{大型の高収益} - \text{大型の低収益}) + \tfrac{1}{2}\,(\text{小型の高収益} - \text{小型の低収益})$$

- 実績と期待で同じ銘柄を使う(両方がそろう銘柄だけ)
- 平均 > 0 の検定(Newey-West t 値と片側 p 値)、期待 − 実績 の差の検定、`data/Japan_5_Factors.csv` へのスパニング回帰

## 0. 設定

In [ ]:
import calendar
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 図の日本語表示

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    display
except NameError:  # Jupyter 以外で実行したとき
    display = print

# ===== 分析期間 =====
START_MONTH = "1995-07"              # 分析開始月(この月以降のファクターリターンで評価)
END_MONTH = None                     # 分析終了月("2024-12" など。None ならデータの最後まで)

# ===== ポートフォリオ =====
FORMATION_MONTH = 6                  # 毎年この月の月末に組成し、翌月から12か月保有
BIG_CUM_SHARE = 0.90                 # 大型:時価総額の大きい順に足して累積 90% に達するまで
BM_BPS = (0.3, 0.7)                  # B/M の分位点
PROF_BPS = (0.3, 0.7)                # 収益性の分位点
WEIGHTING = "vw"                     # "vw":時価総額加重バイアンドホールド、"ew":等ウェイトバイアンドホールド
CUM_METHOD = "index100"              # 図の累積:"index100" = 100 × Π(1 + r)、"simple" = Π(1 + r) − 1、"log" = Σ log(1 + r)
UNIVERSE_TOPIX_ONLY = True           # True:組成日時点の TOPIX 構成銘柄だけを対象にする
EXCLUDE_IND33 = {"4050", "7050", "7100", "7150", "7200"}  # 電気・ガス、銀行、証券、保険、その他金融

# ===== 収益性指標 =====
MEASURES = ["CBOP", "CP", "ROE", "ROA"]
SCALE = {"CBOP": "asset", "ROA": "asset", "CP": "equity", "ROE": "equity"}

# ===== データ =====
MISSING_SENTINEL = -999999998        # IDSQE 系の欠損値(これ以下の値はすべて欠損扱い)
CTYPE_PRIORITY = ["1", "3", "2"]     # 会計基準の優先順位(1: SEC, 3: IFRS, 2: 日本)
CTYPE_NAME = {"1": "SEC", "2": "日本", "3": "IFRS"}
ACCOUNTING_UNIT = 1e6                # 財務データの単位(百万円)→ 円に直す倍率
AVAIL_FALLBACK_MONTHS = 2            # 発表日・短信入手日が欠損のとき、期末 + 2か月で入手可能とみなす
MAX_STALENESS_MONTHS = 15            # 組成時点から見て、期末がこれより古い決算は使わない

# ===== 推定・検定 =====
WINSOR = (0.01, 0.99)
MICRO_PCT = 0.20                     # 期待収益性の推定から除くマイクロキャップ(ユニバースの時価総額 20 パーセンタイル未満)
MIN_OBS_CS = 30
NW_LAGS = 12

FF_PATH = Path("data/Japan_5_Factors.csv")
CACHE_PATH = Path("cache_raw_data.pkl")
OUTPUT_DIR = Path("outputs_rmw18")
OUTPUT_DIR.mkdir(exist_ok=True)

## 0-1. DB 接続

`temp1.ipynb` で使っている DB 接続用の関数 `fetch_data` と `get_data` を、このセルで import するか定義してください。
キャッシュ(`cache_raw_data.pkl`)がある場合は DB に接続しないので、未定義でも実行できます。

In [ ]:
# 例:from db_utils import fetch_data, get_data
# ↑ temp1.ipynb と同じ import 文・定義をここに書く

if "fetch_data" not in globals() or "get_data" not in globals():
    print("[注意] fetch_data / get_data が未定義です。キャッシュがない場合、データ取得でエラーになります。")

## 0-2. データ取得関数

`temp1.ipynb` の取得関数をそのまま載せています(`get_topix_data` の SELECT のカンマ抜けだけ修正)。
欠損値(-999999998)の置き換えは1章で行います。

In [ ]:
# PL項目の取得
def get_pl_data():
    PL_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[GPLS]         -- 売上総利益
        ,[OPINCM]       -- 営業利益
        ,[CHAGE]        -- 支払利息
        ,[PLCURREN]     -- 為替換算調整勘定，のちのちの分析で必要かもなので一応
        ,[NETINCM]      -- 当期純利益
        ,[RD]           -- 研究開発費
    FROM [IDSQE].[dbo].[NRIFIN_CON_PL4]
    """
    df = fetch_data(PL_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# BS項目の取得
def get_bs_data():
    BS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]        -- 決算期 (int yyyymm)
        ,[CTYPE]        -- 会計基準 (1: SEC 2: 日本式 3: IFRS)
        ,[CASH]         -- 現金・預金，のちのちの分析で必要かもなので一応
        ,[ACRC]         -- 売上債権，アクルーアルの計算要
        ,[INVENT]       -- 棚卸資産
        ,[INVSCRT]      -- 投資有価証券，別の分析でのちほど用いる
        ,[TOTALBS]      -- 資産合計
        ,[ACPY]         -- 買入債務
        ,[CAPSTTL]      -- 株主資本合計
        ,[NTASTTL]      -- 純資産合計，株主資本合計が欠損しているときに用いる
        ,[EQTY]         -- 資本合計，株主資本合計が欠損しているとき
    FROM [IDSQE].[dbo].[NRIFIN_CON_BS4]
    """
    df = fetch_data(BS_QUERY, 'idsqe')
    # 欠損値は-999999998
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 決算詳細
def get_accounting_details():
    DETAILS_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FTERM]            -- 決算期 int yyyymm
        ,[CTYPE]            -- 会計基準 1: SEC式 2: 日本式 3: IFRS式
        ,[SETTLEMENT_FLG]   -- 本中区分 1: 本決算 2: 中間決算 3: 第1四半期決算 4: 第3四半期決算
        ,[FDATE]            -- 決算年月日 char yyyymmdd
        ,[NMONTH]           -- 決算月数 通常12
        ,[ANNOUNCE]         -- 決算発表日，[RCVD]より欠損(-999999998)が多い
        ,[RCVD]             -- 決算短信入手日
    FROM [IDSQE].[dbo].[NRIFIN_CON_AT4]
    """
    df = fetch_data(DETAILS_QUERY, 'idsqe')
    # 欠損値は-999999998
    # 決算発表、短信入手日どちらも欠損している場合は、決算期プラス2か月する（日本の法律）
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 会社属性
# NRI_CODE -> 証券コード
def get_nri_to_ticker_table():
    TABLE_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,[ITEM_DATA]        -- 証券コード
    FROM [IDSQE].[dbo].[COMPANY_CODE]
    WHERE ITEM = 'OLDCD'            -- 証券コード
    """
    df = fetch_data(TABLE_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# 業種コード
def get_industry():
    IND_QUERY = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]        -- int yyyymmdd
        ,[TO_DATE]          -- int yyyymmdd
        ,c.[ITEM_DATA]
        ,n.[IND_NAME]
    FROM [IDSQE].[dbo].[COMPANY_INDUSTRY_CLASS] c
    JOIN [IDSQE].[dbo].[INDUSTRY_CLASS_NAME] n ON c.ITEM = n.IND
    WHERE [ITEM] = 'IND33'   -- 東証33業種
    """
    df = fetch_data(IND_QUERY, 'idsqe')
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df


# TOPIX構成銘柄
def get_topix_data():
    query = """
    SELECT
        [NRI_CODE]
        ,[FROM_DATE]
        ,[TO_DATE]
        ,[ITEM]
        ,[ITEM_DATA]
    FROM [IDSQE].[dbo].[STOCK_ISSUE_INDEX]
    WHERE [ITEM] = 'TOPIX';
    """
    return get_data(query, "idsqe")


# 時価総額の取得
def get_market_cap_data():
    # 月次時価総額と、各月末時点で有効な証券コードを取得
    query = """
    SELECT
        prc.[DATE] AS [date],
        prc.[CAP] AS market_cap,
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_PRC] AS prc
    CROSS APPLY (
        -- YYYYMMから月末日（YYYYMMDD）を作成
        SELECT CONVERT(
            int,
            CONVERT(
                char(8),
                EOMONTH(DATEFROMPARTS(
                    prc.[DATE] / 100,
                    prc.[DATE] % 100,
                    1
                )),
                112
            )
        ) AS month_end
    ) AS d
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON prc.[BID] = id.[BID]
        AND id.[IDTYPE] = 'LOCALID'
        AND id.[FDATE] <= d.[month_end]
        AND (id.[TDATE] >= d.[month_end] OR id.[TDATE] IS NULL);
    """
    df = get_data(query, "riskmodels")

    # 証券コードと数値列を整形
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype(int)
    df["market_cap"] = (
        pd.to_numeric(df["market_cap"], errors="raise")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )
    # 月次Period列を追加
    df["month"] = pd.to_datetime(df["date"].astype(str), format="%Y%m").dt.to_period("M")
    return df


# 銘柄リターンの取得
def get_return_data():
    # 月次リターンと、各月初時点で有効な証券コードを取得
    query = """
    SELECT
        rtn.[DATE],
        SUBSTRING(RTRIM(id.[ID]), 3, LEN(RTRIM(id.[ID])))
            AS security_code,
        rtn.[RTN]
    FROM [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_M_RTN] AS rtn
    INNER JOIN [RISK_MODELS].[irddb_rskmdl_owner01].[JPE4_ID] AS id
        ON rtn.[BID] = id.[BID]
    WHERE id.[IDTYPE] = 'LOCALID'
      AND TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
          >= TRY_CONVERT(date, CONVERT(varchar(8), id.[FDATE]), 112)
      AND (
          id.[TDATE] IS NULL
          OR TRY_CONVERT(date, CONCAT(rtn.[DATE], '01'), 112)
             <= TRY_CONVERT(date, CONVERT(varchar(8), id.[TDATE]), 112)
      );
    """
    df = get_data(query, "riskmodels")

    # 証券コード・年月を整形し、リターンを%から小数へ変換
    df["security_code"] = df["security_code"].astype("string").str.strip().str.upper()
    dates = pd.to_numeric(df["DATE"], errors="raise").astype(int)
    df["month"] = pd.to_datetime(dates.astype(str), format="%Y%m").dt.to_period("M")
    df["ret"] = pd.to_numeric(df["RTN"], errors="raise").astype(float) / 100
    return df[["security_code", "month", "ret"]].copy()

## 0-3. データ読み込み

上の取得関数で DB からデータを取得し、キャッシュ(`cache_raw_data.pkl`)に保存します。
2回目以降はキャッシュを読みます。取り直したいときは `USE_CACHE = False` にしてください。

In [ ]:
USE_CACHE = True


def load_raw_data(use_cache=USE_CACHE):
    if "取得データ" in globals():  # temp1.ipynb の取得結果がカーネルに残っている場合
        print("既存の `取得データ` を使います")
        return globals()["取得データ"]
    if use_cache and CACHE_PATH.exists():
        print(f"キャッシュ {CACHE_PATH} を読み込みます")
        with open(CACHE_PATH, "rb") as f:
            return pickle.load(f)
    funcs = {
        "PL": get_pl_data, "BS": get_bs_data, "決算詳細": get_accounting_details,
        "証券コード対応表": get_nri_to_ticker_table, "業種": get_industry,
        "TOPIX構成銘柄": get_topix_data, "時価総額": get_market_cap_data, "月次リターン": get_return_data,
    }
    data = {}
    for name, fn in funcs.items():
        print(f"取得中:{name}")
        data[name] = fn()
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(data, f)
    return data


raw = load_raw_data()
for k, v in raw.items():
    print(f"{k}: {v.shape}")

### 共通の関数

In [ ]:
def clean_idsqe(df):
    """IDSQE 系テーブルの整形:欠損値(-999999998 以下)を NaN に、文字列の前後空白を除去"""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].mask(df[c] <= MISSING_SENTINEL)
        elif pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df


def norm_code(s):
    """'2'・2・2.0 のどれで来ても '2' という文字列にそろえる"""
    num = pd.to_numeric(s, errors="coerce")
    out = s.astype(str).str.strip()
    return out.where(num.isna(), num.round().astype("Int64").astype(str))


def norm_sec(s):
    return s.astype(str).str.strip().str.upper()


def to_date(s):
    """yyyymmdd の整数 → Timestamp(0・99999999・欠損は NaT)"""
    v = pd.to_numeric(s, errors="coerce")
    v = v.where((v > 19000000) & (v < 21000000))
    return pd.to_datetime(v.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")


def to_midx(s):
    """月(Period・'YYYY-MM'・yyyymm 整数)→ 通し月番号 year*12 + month - 1"""
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype(int)
        return (v // 100) * 12 + v % 100 - 1
    dt = pd.to_datetime(s.astype(str))
    return dt.dt.year * 12 + dt.dt.month - 1


def month_str_to_midx(m):
    ts = pd.Timestamp(m)
    return ts.year * 12 + ts.month - 1


def midx_to_ts(m):
    m = np.asarray(m, dtype=int)
    return pd.to_datetime({"year": m // 12, "month": m % 12 + 1, "day": 1}) + pd.offsets.MonthEnd(0)


def formation_date(y):
    return pd.Timestamp(int(y), FORMATION_MONTH, calendar.monthrange(int(y), FORMATION_MONTH)[1])


def formation_int(y):
    d = formation_date(y)
    return d.year * 10000 + d.month * 100 + d.day


def winsorize_by(df, cols, by, limits=WINSOR):
    df = df.copy()
    for c in cols:
        g = df.groupby(by)[c]
        lo = g.transform(lambda x: x.quantile(limits[0]))
        hi = g.transform(lambda x: x.quantile(limits[1]))
        df[c] = df[c].clip(lo, hi)
    return df


def ols_nw(y, X, lags=NW_LAGS):
    """OLS(定数項あり)と Newey-West の t 値。X が (T, 0) なら平均の検定"""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float).reshape(len(y), -1)
    X = np.column_stack([np.ones(len(y)), X])
    ok = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    y, X = y[ok], X[ok]
    T = len(y)
    if T <= X.shape[1] + 1:
        return np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan), T
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for l in range(1, min(lags, T - 1) + 1):
        w = 1 - l / (lags + 1)
        G = Xu[l:].T @ Xu[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    return b, b / np.sqrt(np.diag(V)), T


def mean_t(x, lags=NW_LAGS):
    x = pd.Series(x).dropna()
    b, t, T = ols_nw(x.values, np.empty((len(x), 0)), lags)
    return b[0], t[0], T


def p_one_sided(t):
    """平均 > 0 の片側 p 値(正規近似)"""
    return 0.5 * math.erfc(t / math.sqrt(2)) if pd.notna(t) else np.nan


def p_two_sided(t):
    return math.erfc(abs(t) / math.sqrt(2)) if pd.notna(t) else np.nan


def ols_cs(y, X):
    Xc = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    return b


def asof_lookup(keys, table, value_cols, date_col):
    """NRI_CODE と日付(yyyymmdd 整数)で、FROM_DATE ≤ 日付 ≤ TO_DATE の行を引く"""
    k = keys[["NRI_CODE", date_col]].drop_duplicates()
    m = k.merge(table, on="NRI_CODE", how="inner")
    m = m[(m["FROM_DATE"] <= m[date_col]) & (m[date_col] <= m["TO_DATE"])]
    m = m.sort_values("FROM_DATE").drop_duplicates(["NRI_CODE", date_col], keep="last")
    return m[["NRI_CODE", date_col] + value_cols]


START_MIDX = month_str_to_midx(START_MONTH)
END_MIDX = month_str_to_midx(END_MONTH) if END_MONTH else None
print(f"分析期間:{START_MONTH} 〜 {END_MONTH or 'データの最後'}")

## 1. 財務データの整形と収益性指標の計算

1. 欠損値を NaN に置き換え、**本決算・決算月数12** の行だけを残す
2. PL・BS・決算詳細を (NRI_CODE, 決算期, 会計基準) で結合(会計基準ごとに別の行のまま)
3. 前期・翌期の値は (NRI_CODE, 会計基準, 決算期 ± 12か月) で結合 → 分子と分母の会計基準が一致
4. 入手可能日から、その決算を使う組成年(6月末までに公表されていればその年)を決める

In [ ]:
KEY = ["NRI_CODE", "FTERM", "CTYPE"]

pl = clean_idsqe(raw["PL"])
bs = clean_idsqe(raw["BS"])
at = clean_idsqe(raw["決算詳細"])
for d in (pl, bs, at):
    d["NRI_CODE"] = norm_sec(d["NRI_CODE"])
    d["CTYPE"] = norm_code(d["CTYPE"])
    d["FTERM"] = pd.to_numeric(d["FTERM"]).astype(int)
at["SETTLEMENT_FLG"] = norm_code(at["SETTLEMENT_FLG"])
at = at[(at["SETTLEMENT_FLG"] == "1") & (at["NMONTH"] == 12)]


def dedupe(df, name):
    n = df.duplicated(KEY).sum()
    if n:
        print(f"[注意] {name}: (NRI_CODE, 決算期, 会計基準) の重複が {n} 行 → 先頭の行を残します")
        df = df.drop_duplicates(KEY)
    return df


at, pl, bs = dedupe(at, "決算詳細"), dedupe(pl, "PL"), dedupe(bs, "BS")
acct = at.merge(pl, on=KEY, how="inner").merge(bs, on=KEY, how="inner")
print(f"本決算・12か月の財務データ:{len(acct):,} 行")

# 決算期、期末日、入手可能日
acct["fy_idx"] = (acct["FTERM"] // 100) * 12 + acct["FTERM"] % 100 - 1
fterm_end = midx_to_ts(acct["fy_idx"].to_numpy())
acct["fdate"] = to_date(acct["FDATE"]).fillna(pd.Series(fterm_end.values, index=acct.index))
ann = to_date(acct["ANNOUNCE"])
rcv = to_date(acct["RCVD"])
fallback = acct["fdate"] + pd.offsets.MonthEnd(AVAIL_FALLBACK_MONTHS)
acct["avail_date"] = ann.where(ann >= acct["fdate"]).fillna(rcv.where(rcv >= acct["fdate"])).fillna(fallback)

# 自己資本・総資産
acct["TA"] = acct["TOTALBS"].where(acct["TOTALBS"] > 0)
be_raw = acct["CAPSTTL"].fillna(acct["EQTY"]).fillna(acct["NTASTTL"])
acct["BE"] = be_raw.where(be_raw > 0)

# 前期(同じ会計基準)の値
PREV_COLS = ["TA", "BE", "CAPSTTL", "EQTY", "NTASTTL", "ACRC", "INVENT", "ACPY"]
prev = acct[["NRI_CODE", "CTYPE", "fy_idx"] + PREV_COLS].copy()
prev["fy_idx"] += 12
prev["prev_exists"] = True
acct = acct.merge(prev.rename(columns={c: c + "_p" for c in PREV_COLS}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
acct["prev_exists"] = acct["prev_exists"].eq(True)


def delta_item(cur, prv, exists):
    d = cur - prv
    d = d.mask(cur.isna() & prv.isna(), 0.0)
    return d.where(exists)


acct["accrual"] = (delta_item(acct["ACRC"], acct["ACRC_p"], acct["prev_exists"])
                   + delta_item(acct["INVENT"], acct["INVENT_p"], acct["prev_exists"])
                   - delta_item(acct["ACPY"], acct["ACPY_p"], acct["prev_exists"]))
int_exp = acct["CHAGE"].fillna(0.0)
rd = acct["RD"].fillna(0.0)
numer = {
    "CBOP": acct["OPINCM"] + rd - acct["accrual"],
    "CP": acct["OPINCM"] - int_exp - acct["accrual"],
    "ROE": acct["NETINCM"],
    "ROA": acct["OPINCM"],
}
for m in MEASURES:
    denom = acct["TA_p"] if SCALE[m] == "asset" else acct["BE_p"]
    acct[m] = numer[m] / denom.where(denom > 0)

# 成長率
acct["g_TA"] = acct["TA"] / acct["TA_p"] - 1
# 自己資本:まず両期で同じ項目(株主資本合計どうし、資本合計どうし、純資産合計どうし)で計算
g_be = pd.Series(np.nan, index=acct.index)
for col in ["CAPSTTL", "EQTY", "NTASTTL"]:
    ok = (acct[col] > 0) & (acct[col + "_p"] > 0) & g_be.isna()
    g_be[ok] = acct.loc[ok, col] / acct.loc[ok, col + "_p"] - 1
# 前期と同じ項目がそろわない年(2006年の会社法施行で「資本合計」→「株主資本合計」に切り替わった年など)は、
# 自己資本 BE(株主資本合計 → 資本合計 → 純資産合計 の順で選んだ値)どうしで計算する
fallback = (acct["BE"] / acct["BE_p"] - 1).where(acct["BE"].notna() & acct["BE_p"].notna())
acct["g_BE_fallback"] = g_be.isna() & fallback.notna()   # 代わりの計算を使った行(確認用)
acct["g_BE"] = g_be.fillna(fallback)
print("自己資本成長率を BE どうしで代わりに計算した件数(決算期末の年ごと、上位5年)")
display(acct.loc[acct["g_BE_fallback"]].groupby(acct["fdate"].dt.year).size().nlargest(5).to_frame("件数").T)

# 1期前と1期後(同じ会計基準)
LAG_COLS = MEASURES + ["g_TA", "g_BE"]
lag = acct[["NRI_CODE", "CTYPE", "fy_idx"] + LAG_COLS].copy()
lag["fy_idx"] += 12
acct = acct.merge(lag.rename(columns={c: c + "_lag" for c in LAG_COLS}), on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
lead = acct[["NRI_CODE", "CTYPE", "fy_idx", "avail_date"] + MEASURES].copy()
lead["fy_idx"] -= 12
acct = acct.merge(lead.rename(columns={c: c + "_lead" for c in MEASURES + ["avail_date"]}),
                  on=["NRI_CODE", "CTYPE", "fy_idx"], how="left")
for m in MEASURES:
    acct["d" + m] = acct[m] - acct[m + "_lag"]

# 組成年:入手可能日が組成月末までなら、その年の組成で使える
acct = acct[acct["avail_date"].notna()].copy()
av = acct["avail_date"]
acct["form_year"] = (av.dt.year + (av.dt.month > FORMATION_MONTH).astype(int)).astype(int)
acct["stale_ok"] = (acct["form_year"] * 12 + FORMATION_MONTH - 1 - acct["fy_idx"]) <= MAX_STALENESS_MONTHS
acct["prio"] = acct["CTYPE"].map({c: i for i, c in enumerate(CTYPE_PRIORITY)}).fillna(99)

print("指標ごとの計算できた件数(会計基準別)")
display(acct.groupby(acct["CTYPE"].map(CTYPE_NAME).fillna(acct["CTYPE"]))[MEASURES].count().T)

## 2. 組成年への割り当て・会計基準の選択・市場データとの結合

- 各企業・各組成年について、使える決算のうち最新の決算期を選び、その中で必要な変数(収益性・その変化・投資・自己資本)が
  **同じ会計基準ですべて計算できる** 行を、会計基準の優先順位で1つ選ぶ(指標ごと)
- 組成日(6月末)時点の証券コード・業種・TOPIX 構成を引き、6月末の時価総額と決算期末の時価総額を付ける
- ユニバース:TOPIX 構成銘柄(`UNIVERSE_TOPIX_ONLY`)のうち、電気・ガス・金融を除く銘柄

In [ ]:
code_tbl = clean_idsqe(raw["証券コード対応表"])
code_tbl["NRI_CODE"] = norm_sec(code_tbl["NRI_CODE"])
code_tbl["sec"] = norm_sec(code_tbl["ITEM_DATA"])
code_tbl["FROM_DATE"] = code_tbl["FROM_DATE"].fillna(0)
code_tbl["TO_DATE"] = code_tbl["TO_DATE"].fillna(99999999)

ind_tbl = clean_idsqe(raw["業種"])
ind_tbl["NRI_CODE"] = norm_sec(ind_tbl["NRI_CODE"])
ind_tbl["ind33"] = norm_code(ind_tbl["ITEM_DATA"]).str.zfill(4)
ind_tbl = ind_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE", "ind33"]].drop_duplicates()
ind_tbl["FROM_DATE"] = ind_tbl["FROM_DATE"].fillna(0)
ind_tbl["TO_DATE"] = ind_tbl["TO_DATE"].fillna(99999999)

topix_tbl = clean_idsqe(raw["TOPIX構成銘柄"])
topix_tbl["NRI_CODE"] = norm_sec(topix_tbl["NRI_CODE"])
topix_tbl = topix_tbl[["NRI_CODE", "FROM_DATE", "TO_DATE"]].drop_duplicates()
topix_tbl["FROM_DATE"] = topix_tbl["FROM_DATE"].fillna(0)
topix_tbl["TO_DATE"] = topix_tbl["TO_DATE"].fillna(99999999)
topix_tbl["topix"] = True

mcap = raw["時価総額"].copy()
mcap["sec"] = norm_sec(mcap["security_code"])
mcap["midx"] = to_midx(mcap["month"]) if "month" in mcap else to_midx(mcap["date"])
mcap["me"] = pd.to_numeric(mcap["market_cap"], errors="coerce")
mcap = mcap.loc[mcap["me"] > 0, ["sec", "midx", "me"]].drop_duplicates(["sec", "midx"])

ret = raw["月次リターン"].copy()
ret["sec"] = norm_sec(ret["security_code"])
ret["midx"] = to_midx(ret["month"])
ret["ret"] = pd.to_numeric(ret["ret"], errors="coerce")
ret = ret.loc[ret["ret"].notna(), ["sec", "midx", "ret"]].drop_duplicates(["sec", "midx"])


def select_records(req_cols):
    d = acct[acct["stale_ok"]].dropna(subset=req_cols)
    d = d.sort_values(["NRI_CODE", "form_year", "fy_idx", "prio"], ascending=[True, True, False, True])
    return d.drop_duplicates(["NRI_CODE", "form_year"], keep="first").copy()


def attach_market(d):
    d = d.copy()
    d["F_int"] = d["form_year"].map(formation_int)
    d = d.merge(asof_lookup(d, code_tbl, ["sec"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, ind_tbl, ["ind33"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d = d.merge(asof_lookup(d, topix_tbl, ["topix"], "F_int"), on=["NRI_CODE", "F_int"], how="left")
    d["topix"] = d["topix"].eq(True)
    d["midx_form"] = d["form_year"] * 12 + FORMATION_MONTH - 1
    d = d.merge(mcap.rename(columns={"midx": "midx_form", "me": "ME_form"}), on=["sec", "midx_form"], how="left")
    d = d.merge(mcap.rename(columns={"midx": "fy_idx", "me": "ME_fye"}), on=["sec", "fy_idx"], how="left")
    d = d[d["sec"].notna() & (d["ME_form"] > 0) & d["ind33"].notna() & ~d["ind33"].isin(EXCLUDE_IND33)].copy()
    if UNIVERSE_TOPIX_ONLY:
        d = d[d["topix"]].copy()
    d["lnME"] = np.log(d["ME_form"])
    bm = d["BE"] * ACCOUNTING_UNIT / d["ME_fye"]
    d["lnBM"] = np.log(bm.where(bm > 0))
    d["micro"] = d["ME_form"] < d.groupby("form_year")["ME_form"].transform(lambda s: s.quantile(MICRO_PCT))
    return d.drop_duplicates(["NRI_CODE", "form_year"])


def build_measure_panel(m):
    inv_col = "g_TA_lag" if SCALE[m] == "asset" else "g_BE_lag"
    d = select_records([m, "d" + m, inv_col, "BE"])
    d = d.rename(columns={m: "Y", "d" + m: "dY", m + "_lead": "Y_lead",
                          "avail_date_lead": "avail_lead", inv_col: "inv"})
    keep = ["NRI_CODE", "form_year", "CTYPE", "fy_idx", "BE", "Y", "dY", "Y_lead", "avail_lead", "inv"]
    d = attach_market(d[keep]).dropna(subset=["lnBM"])
    return winsorize_by(d, ["Y", "dY", "inv", "lnBM"], "form_year")


panels = {}
for m in MEASURES:
    panels[m] = build_measure_panel(m)
    p = panels[m]
    print(f"{m}: {len(p):,} 企業年、組成年 {p['form_year'].min()}〜{p['form_year'].max()}、"
          f"会計基準 {p['CTYPE'].map(CTYPE_NAME).value_counts().to_dict()}")

## 3. 期待収益性の推定

組成年 y の期待収益性 Yhat は、前年のコホート(y−1 年組成)のうち **y 年6月末までに翌期の実績が公表された企業** で
翌期の収益性を回帰し、その係数を y 年の説明変数に当てはめて作ります(先読みなし)。

下の表は、年ごとの回帰の係数の時系列平均と Newey-West t 値(ラグ3)、決定係数 R²・自由度調整済み R² の平均、平均銘柄数です。

In [ ]:
XCOLS = ["Y", "dY", "lnBM", "lnME", "inv"]
NW_LAGS_ANNUAL = 3


def add_expected(p):
    p = p.copy()
    coef_rows = {}
    for y in sorted(p["form_year"].unique()):
        tr = p[(p["form_year"] == y - 1) & ~p["micro"] & (p["avail_lead"] <= formation_date(y))]
        tr = tr.dropna(subset=XCOLS + ["Y_lead"])
        if len(tr) < MIN_OBS_CS:
            continue
        lo, hi = tr["Y_lead"].quantile(WINSOR)
        yv = tr["Y_lead"].clip(lo, hi).to_numpy(float)
        X = tr[XCOLS].to_numpy(float)
        b = ols_cs(yv, X)
        e = yv - (b[0] + X @ b[1:])
        n, k = len(yv), len(XCOLS) + 1
        r2 = 1 - (e @ e) / ((yv - yv.mean()) @ (yv - yv.mean()))
        coef_rows[y] = np.r_[b, r2, 1 - (1 - r2) * (n - 1) / (n - k), n]
    if not coef_rows:
        raise ValueError("期待収益性を推定できる年がありません")
    coef = pd.DataFrame.from_dict(coef_rows, orient="index",
                                  columns=["const"] + XCOLS + ["R2", "adjR2", "N"]).sort_index()
    p["Yhat"] = np.nan
    for y, b in coef.iterrows():
        mask = p["form_year"] == y
        p.loc[mask, "Yhat"] = b["const"] + p.loc[mask, XCOLS].to_numpy(float) @ b[XCOLS].to_numpy(float)
    return p, coef


first_stage_coef = {}
for m in MEASURES:
    panels[m], first_stage_coef[m] = add_expected(panels[m])

fs_rows = {}
for m, c in first_stage_coef.items():
    r = {}
    for v in ["const"] + XCOLS:
        mu, t, _ = mean_t(c[v], NW_LAGS_ANNUAL)
        r[(v, "係数")] = mu
        r[(v, "t値")] = t
    r[("決定係数", "R2")] = c["R2"].mean()
    r[("決定係数", "adjR2")] = c["adjR2"].mean()
    r[("", "平均N")] = c["N"].mean()
    r[("", "年数")] = len(c)
    fs_rows[m] = r
first_stage_summary = pd.DataFrame(fs_rows).T
print("期待収益性の推定:年次回帰の係数の平均・t 値と決定係数")
display(first_stage_summary)

## 4. 月次データ

月次リターン、モメンタム(R(1,1)・R(2,12))、バイアンドホールドのウェイトを作ります。組成年 y の保有期間は y 年7月〜y+1 年6月です。

In [ ]:
OUTPUT_DIR = Path("outputs_expected_fm")
OUTPUT_DIR.mkdir(exist_ok=True)
GROUPS = {"キャッシュベース(CBOP・CP)": ["CBOP", "CP"], "なじみのある指標(ROA・ROE)": ["ROA", "ROE"]}
CUM_LABEL = "累積リターン指数(開始月の前月末 = 100)"
NW_LAGS_ANNUAL = 3

secs = set()
for p in panels.values():
    secs |= set(p["sec"])
r_wide = ret[ret["sec"].isin(secs)].pivot(index="midx", columns="sec", values="ret")
r_wide = r_wide.reindex(range(r_wide.index.min(), r_wide.index.max() + 1))
r11 = r_wide.shift(1)
r212 = np.expm1(np.log1p(r_wide.clip(lower=-0.99)).shift(2).rolling(11, min_periods=11).sum())
T_, N_ = r_wide.shape
mlong = pd.DataFrame({
    "midx": np.repeat(r_wide.index.to_numpy(), N_),
    "sec": np.tile(r_wide.columns.to_numpy(), T_),
    "ret": r_wide.to_numpy().ravel(),
    "R11": r11.to_numpy().ravel(),
    "R212": r212.to_numpy().ravel(),
})
mlong = mlong[mlong["ret"].notna()].copy()
mlong["form_year"] = (mlong["midx"] - FORMATION_MONTH) // 12
mlong = mlong.sort_values(["sec", "midx"])
gross = (1 + mlong["ret"].clip(lower=-0.99)).groupby([mlong["sec"], mlong["form_year"]]).cumprod()
mlong["growth_before"] = gross.groupby([mlong["sec"], mlong["form_year"]]).shift(1).fillna(1.0)
mlong = mlong[mlong["midx"] >= START_MIDX]
if END_MIDX is not None:
    mlong = mlong[mlong["midx"] <= END_MIDX]
del r_wide, r11, r212
print(f"月次データ:{len(mlong):,} 行、{mlong['midx'].nunique()} か月")


def in_period(g):
    """分析期間に保有期間が重なる組成年だけを残す"""
    g = g[g["form_year"] * 12 + FORMATION_MONTH + 11 >= START_MIDX]
    if END_MIDX is not None:
        g = g[g["form_year"] * 12 + FORMATION_MONTH <= END_MIDX]
    return g


def ols_fit(y, X):
    """クロスセクション OLS:係数・自由度調整済み R²・N"""
    Xc = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    e = y - Xc @ b
    n, k = Xc.shape
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return b, 1 - (1 - r2) * (n - 1) / (n - k), n


def judge(diff, t):
    if pd.isna(diff) or pd.isna(t):
        return ""
    if diff > 0:
        return "改善(有意)" if t >= 1.96 else "改善"
    return "悪化(有意)" if t <= -1.96 else "悪化"

## 5. 将来の収益性の予測

### 5-0. 計算

In [ ]:
def profitability_regressions(p):
    specs = {"(A) Y": ["Y"], "(B) Y, Y(t−1)": ["Y", "Y_lag"], "(C) Y, ΔY": ["Y", "dY"]}
    ctrl = ["lnBM", "lnME", "inv"]
    s = in_period(p[~p["micro"]]).assign(Y_lag=lambda d: d["Y"] - d["dY"])
    s = s.dropna(subset=["Y", "Y_lag", "dY", "Y_lead"] + ctrl)
    rows = {k: {} for k in specs}
    for y, g in s.groupby("form_year"):
        if len(g) < MIN_OBS_CS:
            continue
        lo, hi = g["Y_lead"].quantile(WINSOR)
        yv = g["Y_lead"].clip(lo, hi).to_numpy(float)
        for k, xs in specs.items():
            b, adj, n = ols_fit(yv, g[xs + ctrl].to_numpy(float))
            rows[k][y] = np.r_[b, adj, n]
    out = {}
    for k, xs in specs.items():
        df = pd.DataFrame.from_dict(rows[k], orient="index", columns=["const"] + xs + ctrl + ["adjR2", "N"])
        col = {}
        for v in xs + ctrl:
            mu, t, _ = mean_t(df[v], NW_LAGS_ANNUAL)
            col[(v, "係数")] = mu
            col[(v, "t値")] = t
        col[("平均adjR2", "")] = df["adjR2"].mean()
        col[("平均N", "")] = df["N"].mean()
        col[("年数", "")] = len(df)
        out[k] = pd.Series(col)
    out = pd.DataFrame(out)
    r2a, r2c = out.loc[("平均adjR2", ""), "(A) Y"], out.loc[("平均adjR2", ""), "(C) Y, ΔY"]
    out.loc[("偏R2(C vs A)", ""), "(C) Y, ΔY"] = (r2c - r2a) / (1 - r2a)
    order = [(v, s_) for v in ["Y", "Y_lag", "dY"] + ctrl for s_ in ["係数", "t値"]]
    order += [("平均adjR2", ""), ("偏R2(C vs A)", ""), ("平均N", ""), ("年数", "")]
    return out.reindex(pd.MultiIndex.from_tuples(order))


def forecast_accuracy(p):
    preds = {"実績(ランダムウォーク)": "Y", "期待 Yhat(サンプル外)": "Yhat"}
    s = in_period(p[~p["micro"]]).dropna(subset=["Y_lead", "Y", "Yhat"])
    rows = []
    for y, g in s.groupby("form_year"):
        if len(g) < MIN_OBS_CS:
            continue
        lo, hi = g["Y_lead"].quantile(WINSOR)
        tgt = g["Y_lead"].clip(lo, hi)
        for name, col in preds.items():
            e = tgt - g[col]
            rows.append({"year": y, "予測": name,
                         "順位相関": np.corrcoef(tgt.rank(), g[col].rank())[0, 1],
                         "MAE": e.abs().mean(),
                         "R2(サンプル外)": 1 - (e ** 2).sum() / ((tgt - tgt.mean()) ** 2).sum()})
    py = pd.DataFrame(rows)
    summary = py.groupby("予測")[["順位相関", "MAE", "R2(サンプル外)"]].mean().loc[list(preds)]
    wide = py.pivot(index="year", columns="予測", values="順位相関")
    mu, t, _ = mean_t(wide["期待 Yhat(サンプル外)"] - wide["実績(ランダムウォーク)"], NW_LAGS_ANNUAL)
    summary.loc["期待 Yhat(サンプル外)", "順位相関の差(期待−実績)"] = mu
    summary.loc["期待 Yhat(サンプル外)", "差のt値"] = t
    return summary


table1 = {m: profitability_regressions(panels[m]) for m in MEASURES}
accuracy = {m: forecast_accuracy(panels[m]) for m in MEASURES}


def show_forecast_group(ms):
    print("■ 将来の収益性の予測回帰(年次回帰の係数の平均・t 値、自由度調整済み R² の平均)")
    display(pd.concat({m: table1[m] for m in ms}, axis=1))
    print("■ 予測精度(毎年の値の平均。マイクロキャップ除く)")
    display(pd.concat({m: accuracy[m] for m in ms}))

### 5-1. キャッシュベース(CBOP・CP)

In [ ]:
show_forecast_group(GROUPS["キャッシュベース(CBOP・CP)"])

### 5-2. なじみのある指標(ROA・ROE)

ROA・ROE が過去の情報(当期の収益性・その変化・B/M・規模・投資)からどれだけ予測できるかを、R² と偏 R²、予測精度で確認します。

In [ ]:
show_forecast_group(GROUPS["なじみのある指標(ROA・ROE)"])

In [ ]:
# 図:予測の説明力(R²)と順位相関を4指標で比べる
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(MEASURES))
r2A = [table1[m].loc[("平均adjR2", ""), "(A) Y"] for m in MEASURES]
r2C = [table1[m].loc[("平均adjR2", ""), "(C) Y, ΔY"] for m in MEASURES]
axes[0].bar(x - 0.2, r2A, 0.4, label="(A) Y + コントロール")
axes[0].bar(x + 0.2, r2C, 0.4, label="(C) Y, ΔY + コントロール")
axes[0].set_title("翌期の収益性の予測:自由度調整済み R² の平均")
rc_y = [accuracy[m].loc["実績(ランダムウォーク)", "順位相関"] for m in MEASURES]
rc_h = [accuracy[m].loc["期待 Yhat(サンプル外)", "順位相関"] for m in MEASURES]
axes[1].bar(x - 0.2, rc_y, 0.4, label="実績 Y")
axes[1].bar(x + 0.2, rc_h, 0.4, label="期待 Yhat")
axes[1].set_title("翌期の収益性との順位相関(サンプル外)")
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(MEASURES)
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "forecast_r2_rankcorr.png", dpi=120)
plt.show()

## 6. Fama-MacBeth 回帰

### 6-0. 計算

In [ ]:
def fama_macbeth(df, ycol, xcols, wcol=None, tcol="midx", min_obs=MIN_OBS_CS):
    """毎月のクロスセクション回帰(wcol があれば WLS)。coef・n・adjr2・iqr を返す"""
    cols = [tcol, ycol] + xcols + ([wcol] if wcol else [])
    d = df[cols].dropna().sort_values(tcol)
    if wcol:
        d = d[d[wcol] > 0]
    t = d[tcol].to_numpy()
    Y = d[ycol].to_numpy(float)
    X = d[xcols].to_numpy(float)
    W = d[wcol].to_numpy(float) if wcol else None
    cuts = np.flatnonzero(np.diff(t)) + 1
    rows, iqr_rows = {}, {}
    for s, e in zip(np.r_[0, cuts], np.r_[cuts, len(t)]):
        n = e - s
        if n < max(min_obs, len(xcols) + 5):
            continue
        y_, X_ = Y[s:e], X[s:e]
        Xc = np.column_stack([np.ones(n), X_])
        if W is not None:
            w = W[s:e] / W[s:e].sum()
            sw = np.sqrt(w)
            b, *_ = np.linalg.lstsq(Xc * sw[:, None], y_ * sw, rcond=None)
            res = y_ - Xc @ b
            r2 = 1 - (w * res ** 2).sum() / (w * (y_ - (w * y_).sum()) ** 2).sum()
        else:
            b, *_ = np.linalg.lstsq(Xc, y_, rcond=None)
            res = y_ - Xc @ b
            r2 = 1 - (res @ res) / ((y_ - y_.mean()) @ (y_ - y_.mean()))
        k = Xc.shape[1]
        rows[t[s]] = np.r_[b, n, 1 - (1 - r2) * (n - 1) / (n - k)]
        q75, q25 = np.percentile(X_, [75, 25], axis=0)
        iqr_rows[t[s]] = q75 - q25
    res = pd.DataFrame.from_dict(rows, orient="index", columns=["const"] + xcols + ["n", "adjr2"]).sort_index()
    iqr = pd.DataFrame.from_dict(iqr_rows, orient="index", columns=xcols).sort_index()
    return {"coef": res[["const"] + xcols], "n": res["n"], "adjr2": res["adjr2"], "iqr": iqr}


def summarize_fm(res):
    out = {}
    for c in res["coef"].columns:
        mu, t, _ = mean_t(res["coef"][c])
        out[c] = {"係数(×100)": mu * 100, "t値": t}
    out = pd.DataFrame(out).T
    out.loc["平均N", "係数(×100)"] = res["n"].mean()
    out.loc["平均adjR2", "係数(×100)"] = res["adjr2"].mean()
    return out


CONTROLS = ["lnBM", "lnME", "inv", "R11", "R212"]
SPECS = {"(1) Y": ["Y"], "(2) Y+ΔY": ["Y", "dY"], "(3) Y+Yhat": ["Y", "Yhat"], "(4) Yhat": ["Yhat"]}
UNIVERSES = {"全銘柄": lambda d: d, "マイクロ除く": lambda d: d[~d["micro"]]}
FM_WEIGHTS = {"等加重": None, "時価総額加重": "w_vw"}


def fm_data(m):
    cols = ["NRI_CODE", "sec", "form_year", "micro", "ME_form", "Y", "dY", "Yhat", "lnBM", "lnME", "inv"]
    d = panels[m][cols].merge(mlong, on=["sec", "form_year"], how="inner")
    d["w_vw"] = d["ME_form"] * d["growth_before"]
    return d.dropna(subset=["ret", "Y", "dY", "Yhat", "w_vw"] + CONTROLS)


fm_detail, fm_rows = {}, []
for m in MEASURES:
    d_all = fm_data(m)
    for uname, ufun in UNIVERSES.items():
        d = ufun(d_all)
        for wname, wcol in FM_WEIGHTS.items():
            res = {s: fama_macbeth(d, "ret", xs + CONTROLS, wcol) for s, xs in SPECS.items()}
            fm_detail[(m, uname, wname)] = res
            row = {"指標": m, "ユニバース": uname, "加重": wname}
            for label, spec, var in [("Y (1)", "(1) Y", "Y"), ("ΔY (2)", "(2) Y+ΔY", "dY"),
                                     ("Y (3)", "(3) Y+Yhat", "Y"), ("Yhat (3)", "(3) Y+Yhat", "Yhat"),
                                     ("Yhat (4)", "(4) Yhat", "Yhat")]:
                mu, t, _ = mean_t(res[spec]["coef"][var])
                row[(label, "係数(×100)")] = mu * 100
                row[(label, "t値")] = t
            iqr_y = res["(1) Y"]["coef"]["Y"] * res["(1) Y"]["iqr"]["Y"]
            iqr_h = res["(4) Yhat"]["coef"]["Yhat"] * res["(4) Yhat"]["iqr"]["Yhat"]
            _, t_d, _ = mean_t(iqr_h - iqr_y)
            row[("IQRリターン差(%/月)", "Y (1)")] = iqr_y.mean() * 100
            row[("IQRリターン差(%/月)", "Yhat (4)")] = iqr_h.mean() * 100
            row[("IQRリターン差(%/月)", "差のt値")] = t_d
            row[("", "平均N")] = res["(1) Y"]["n"].mean()
            fm_rows.append(row)
fm_summary = pd.DataFrame(fm_rows).set_index(["指標", "ユニバース", "加重"])
fm_summary.columns = pd.MultiIndex.from_tuples(fm_summary.columns)


def show_fm(m, uname="全銘柄", wname="等加重"):
    res = fm_detail[(m, uname, wname)]
    tbl = pd.concat({s: summarize_fm(r) for s, r in res.items()}, axis=1)
    return tbl.reindex(["Y", "dY", "Yhat"] + CONTROLS + ["const", "平均N", "平均adjR2"])


def show_fm_group(ms):
    print("■ 要約(収益性の変数の係数×100 と t 値、IQR リターン差)")
    display(fm_summary.loc[ms])
    for m in ms:
        print(f"■ {m}:式ごとの詳細(全銘柄・等加重)")
        display(show_fm(m))

### 6-1. キャッシュベース(CBOP・CP)

In [ ]:
show_fm_group(GROUPS["キャッシュベース(CBOP・CP)"])

### 6-2. なじみのある指標(ROA・ROE)

ROA・ROE の実績と期待が、ほかの要因(B/M・規模・投資・モメンタム)をコントロールしたうえで、リターンをどれだけ説明できるかを見ます。

In [ ]:
show_fm_group(GROUPS["なじみのある指標(ROA・ROE)"])

In [ ]:
# 図:実績 Y(式1)と期待 Yhat(式4)の t 値を4指標・ユニバース・加重で比べる
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
combos = [(u, w) for u in UNIVERSES for w in FM_WEIGHTS]
for ax, (u, w) in zip(axes, combos):
    ty = [fm_summary.loc[(m, u, w), ("Y (1)", "t値")] for m in MEASURES]
    th = [fm_summary.loc[(m, u, w), ("Yhat (4)", "t値")] for m in MEASURES]
    x = np.arange(len(MEASURES))
    ax.bar(x - 0.2, ty, 0.4, label="実績 Y(式1)")
    ax.bar(x + 0.2, th, 0.4, label="期待 Yhat(式4)")
    ax.axhline(1.96, color="gray", ls="--", lw=0.8)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(MEASURES)
    ax.set_title(f"{u}・{w}")
axes[0].set_ylabel("係数の t 値")
axes[0].legend()
fig.suptitle("Fama-MacBeth 回帰:収益性の係数の t 値(点線 = 1.96)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fm_tvalues.png", dpi=120)
plt.show()

## 7. RMW6:実績 vs 期待

### 7-0. 計算

In [ ]:
def big_threshold(me, share=BIG_CUM_SHARE):
    s = np.sort(np.asarray(me, dtype=float))[::-1]
    cum_before = np.cumsum(s) - s
    return s[cum_before / s.sum() < share].min()


def tercile(d, col, keys, bps):
    lo = d.groupby(keys)[col].transform(lambda x: x.quantile(bps[0]))
    hi = d.groupby(keys)[col].transform(lambda x: x.quantile(bps[1]))
    return pd.Series(np.where(d[col] <= lo, "L", np.where(d[col] >= hi, "H", "M")), index=d.index)


def make_rmw6(df, signal):
    d = df.dropna(subset=[signal, "ME_form"]).copy()
    thr = d.groupby("form_year")["ME_form"].transform(big_threshold)
    d["size"] = np.where(d["ME_form"] >= thr, "B", "S")
    d["prof"] = tercile(d, signal, ["form_year", "size"], PROF_BPS)
    d["grp"] = d["size"] + "/" + d["prof"]
    x = d[["sec", "form_year", "ME_form", "grp"]].merge(
        mlong[["sec", "form_year", "midx", "ret", "growth_before"]], on=["sec", "form_year"])
    x["w"] = x["ME_form"] * x["growth_before"] if WEIGHTING == "vw" else x["growth_before"]
    x = x[(x["w"] > 0) & x["ret"].notna()]
    x["wr"] = x["ret"] * x["w"]
    g = x.groupby(["midx", "grp"])
    cells = [f"{s}/{q}" for s in "BS" for q in "LMH"]
    rets = (g["wr"].sum() / g["w"].sum()).unstack("grp").reindex(columns=cells)
    counts = g.size().unstack("grp").reindex(columns=cells)
    big = rets["B/H"] - rets["B/L"]
    small = rets["S/H"] - rets["S/L"]
    return {"factor": 0.5 * big + 0.5 * small, "big": big, "small": small, "rets": rets, "counts": counts}


SIGNALS = {"実績": "Y", "期待": "Yhat"}

def load_ff_factors(path):
    """Date(yyyymm)と各ファクター(%)の CSV を読む。説明文や年次の部分があっても月次だけを取り出す"""
    lines = Path(path).read_text(encoding="utf-8-sig", errors="ignore").splitlines()
    start = next(i for i, l in enumerate(lines) if "Mkt-RF" in l)
    cols = [c.strip() for c in lines[start].split(",")][1:]
    rows = []
    for l in lines[start + 1:]:
        parts = [x.strip() for x in l.split(",")]
        if len(parts[0]) != 6 or not parts[0].isdigit():
            if rows:
                break  # 月次部分の終わり
            continue
        rows.append(parts[: len(cols) + 1])
    df = pd.DataFrame(rows, columns=["Date"] + cols)
    vals = df[cols].apply(pd.to_numeric, errors="coerce").mask(lambda x: x <= -99.99) / 100
    vals.index = to_midx(df["Date"].astype(int)).to_numpy()
    return vals


def regress(f, X):
    """f を X に回帰。戻り値:{変数名: (係数, t値)}(切片は 'α(%/月)' で %/月)と、月数・自由度調整済み R²"""
    d = pd.concat([f.rename("f"), X], axis=1).dropna()
    b, tb, T = ols_nw(d["f"].values, d[X.columns].values)
    Xc = np.column_stack([np.ones(len(d)), d[X.columns].values])
    e = d["f"].values - Xc @ b
    y = d["f"].values
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    adj = 1 - (1 - r2) * (T - 1) / (T - Xc.shape[1])
    coefs = {"α(%/月)": (b[0] * 100, tb[0])}
    for i, c in enumerate(X.columns, start=1):
        coefs[c] = (b[i], tb[i])
    return coefs, T, adj


def spanning_table(results):
    """results: {(指標, 実績/期待, ファクター): (coefs, T, adjR2)} → 行=ファクター、列=(説明変数, 係数/t値)"""
    rows = {}
    for key, (coefs, T, adj) in results.items():
        r = {}
        for var, (b, t) in coefs.items():
            r[(var, "係数")] = b
            r[(var, "t値")] = t
        r[("α", "p値(両側)")] = p_two_sided(coefs["α(%/月)"][1])
        r[("", "adjR2")] = adj
        r[("", "月数")] = T
        rows[key] = r
    tbl = pd.DataFrame(rows).T
    tbl.index.names = ["指標", "実績/期待", "ファクター"]
    order = [c for c in tbl.columns if c[0] == "α(%/月)"] + [("α", "p値(両側)")]
    order += [c for c in tbl.columns if c[0] not in ("α(%/月)", "α", "")] + [("", "adjR2"), ("", "月数")]
    return tbl[order]


def factor_stats(f):
    f = f.dropna()
    mu, t, T = mean_t(f)
    p = p_one_sided(t)
    return {"平均(%/月)": mu * 100, "年率(%)": mu * 1200, "t値": t, "p値(片側)": p,
            "年率SR": f.mean() / f.std() * np.sqrt(12), "月数": T, "判定(5%)": "有意に正" if p < 0.05 else "有意でない"}


factors6 = {}
for m in MEASURES:
    p = panels[m].dropna(subset=["Y", "Yhat"])     # 実績と期待で同じ銘柄
    for sname, col in SIGNALS.items():
        factors6[(m, sname)] = make_rmw6(p, col)

rows = []
for (m, s), fac in factors6.items():
    r = {"指標": m, "実績/期待": s, **factor_stats(fac["factor"])}
    for leg, lab in [("big", "大型"), ("small", "小型")]:
        mu, t, _ = mean_t(fac[leg])
        r[f"{lab} 平均(%/月)"] = mu * 100
        r[f"{lab} t値"] = t
    rows.append(r)
rmw6_table = pd.DataFrame(rows).set_index(["指標", "実績/期待"])

diff_rows = []
for m in MEASURES:
    for part, lab in [("factor", "RMW6"), ("big", "大型レッグ"), ("small", "小型レッグ")]:
        d = factors6[(m, "期待")][part] - factors6[(m, "実績")][part]
        mu, t, T = mean_t(d)
        diff_rows.append({"指標": m, "比較": f"{lab}:期待 − 実績", "差(%/月)": mu * 100, "t値": t,
                          "p値(両側)": p_two_sided(t), "判定": judge(mu, t), "月数": T})
rmw6_diff = pd.DataFrame(diff_rows).set_index(["指標", "比較"])

# スパニング回帰
if FF_PATH.exists():
    ff = load_ff_factors(FF_PATH)
    FF_COLS = [c for c in ["Mkt-RF", "SMB", "HML", "RMW", "CMA"] if c in ff.columns]
    span = {(m, s): regress(fac["factor"], ff[FF_COLS]) for (m, s), fac in factors6.items()}
    rmw6_spanning = spanning_table({(m, s, "RMW6"): v for (m, s), v in span.items()})
else:
    rmw6_spanning = None
    print(f"[注意] {FF_PATH} が見つかりません。スパニング回帰は飛ばします。")


def cum_index(f):
    f = f.dropna()
    x = midx_to_ts(f.index.to_numpy())
    x0 = x.iloc[0] - pd.offsets.MonthEnd(1)
    return (pd.concat([pd.Series([x0]), x]).reset_index(drop=True),
            pd.concat([pd.Series([100.0]), 100 * (1 + f).cumprod()]).reset_index(drop=True))


def show_rmw6_group(gname, ms):
    print("■ RMW6 のファクターリターン(平均 > 0 の検定)")
    display(rmw6_table.loc[ms])
    print("■ 期待 − 実績 の差の検定")
    display(rmw6_diff.loc[ms])
    if rmw6_spanning is not None:
        print(f"■ スパニング回帰(説明変数 = {' + '.join(FF_COLS)}、係数と Newey-West t 値)")
        display(rmw6_spanning.loc[ms])
    fig, axes = plt.subplots(1, len(ms), figsize=(7 * len(ms), 4.5), sharey=True)
    for ax, m in zip(np.atleast_1d(axes), ms):
        for s, color in [("実績", "tab:blue"), ("期待", "tab:orange")]:
            x, y = cum_index(factors6[(m, s)]["factor"])
            ax.plot(x, y, color=color, label=f"{s}")
        ax.axhline(100, color="gray", lw=0.5)
        ax.set_title(f"RMW6・{m}")
        ax.legend()
    fig.suptitle(f"{gname}:RMW6 の実績 vs 期待({CUM_LABEL})")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"rmw6_cum_{'_'.join(ms)}.png", dpi=120)
    plt.show()

### 7-1. キャッシュベース(CBOP・CP)

In [ ]:
show_rmw6_group("キャッシュベース(CBOP・CP)", GROUPS["キャッシュベース(CBOP・CP)"])

### 7-2. なじみのある指標(ROA・ROE)

In [ ]:
show_rmw6_group("なじみのある指標(ROA・ROE)", GROUPS["なじみのある指標(ROA・ROE)"])

## 8. まとめ

指標ごとに、予測・リターンの説明・ファクターの3つの観点で、実績と期待を並べます。
「期待の効果」は、期待 − 実績 の差の符号と t 値から「改善(有意)/改善/悪化/悪化(有意)」と判定します(|t| ≥ 1.96 で有意)。

In [ ]:
sum_rows = []
for gname, ms in GROUPS.items():
    for m in ms:
        r = {"グループ": gname, "指標": m}
        t1, acc = table1[m], accuracy[m]
        r[("予測", "adjR2 (C)")] = t1.loc[("平均adjR2", ""), "(C) Y, ΔY"]
        r[("予測", "偏R2(ΔY)")] = t1.loc[("偏R2(C vs A)", ""), "(C) Y, ΔY"]
        r[("予測", "順位相関 実績")] = acc.loc["実績(ランダムウォーク)", "順位相関"]
        r[("予測", "順位相関 期待")] = acc.loc["期待 Yhat(サンプル外)", "順位相関"]
        r[("予測", "期待の効果")] = judge(acc.loc["期待 Yhat(サンプル外)", "順位相関の差(期待−実績)"],
                                     acc.loc["期待 Yhat(サンプル外)", "差のt値"])
        for u, w, lab in [("全銘柄", "等加重", "FM 全・等"), ("マイクロ除く", "時価総額加重", "FM 大中・VW")]:
            s = fm_summary.loc[(m, u, w)]
            r[(lab, "Y t値")] = s[("Y (1)", "t値")]
            r[(lab, "Yhat t値")] = s[("Yhat (4)", "t値")]
            r[(lab, "期待の効果")] = judge(s[("IQRリターン差(%/月)", "Yhat (4)")] - s[("IQRリターン差(%/月)", "Y (1)")],
                                       s[("IQRリターン差(%/月)", "差のt値")])
        for sname in SIGNALS:
            r[("RMW6", f"{sname} 平均(%/月)")] = rmw6_table.loc[(m, sname), "平均(%/月)"]
            r[("RMW6", f"{sname} t値")] = rmw6_table.loc[(m, sname), "t値"]
        r[("RMW6", "期待の効果")] = rmw6_diff.loc[(m, "RMW6:期待 − 実績"), "判定"]
        sum_rows.append(r)
summary = pd.DataFrame(sum_rows).set_index(["グループ", "指標"])
summary.columns = pd.MultiIndex.from_tuples(summary.columns)
display(summary)

## 9. 結果の保存

In [ ]:
def save(df, name):
    df.to_csv(OUTPUT_DIR / f"{name}.csv", encoding="utf-8-sig")


for m in MEASURES:
    save(table1[m], f"5_profitability_regression_{m}")
    save(accuracy[m], f"5_forecast_accuracy_{m}")
    save(show_fm(m), f"6_fama_macbeth_{m}_全銘柄_等加重")
save(fm_summary, "6_fama_macbeth_summary")
save(rmw6_table, "7_rmw6_stats")
save(rmw6_diff, "7_rmw6_expected_minus_actual")
if rmw6_spanning is not None:
    save(rmw6_spanning, "7_rmw6_spanning")
fr = pd.DataFrame({f"{m}|{s}": fac["factor"] for (m, s), fac in factors6.items()}).sort_index()
fr.index = midx_to_ts(fr.index.to_numpy()).dt.strftime("%Y-%m").values
save(fr, "7_rmw6_returns_monthly")
save(summary, "8_summary")
print(f"保存先:{OUTPUT_DIR.resolve()}")